# Dynamic Batching Under Concurrent Inference

### Why a larger batch can make serving slower

**Question.** Under concurrent inference, how should a batcher trade queueing delay against accelerator utilization?

This notebook implements the batcher, load generator, instrumentation, and analysis directly. It deliberately preserves a failed first configuration because the failure reveals the scheduling rule more clearly than a successful demo.


## 1 — Hypothesis

Batching should improve throughput while the accelerator is under-filled, but only if enough requests arrive before the queue-delay budget expires.

A necessary bound is:

```text
realized_batch_size <= min(configured_max_batch, offered_concurrency, queued_requests)
```

Therefore setting `max_batch` above offered concurrency cannot create a larger instantaneous batch. It can only increase the chance that the worker waits for requests that cannot arrive in the current wave.


In [ ]:
import time, queue, threading, statistics
from dataclasses import dataclass, field
from typing import Any
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "CUDA device required to rerun this benchmark"
device = torch.device("cuda")
torch.manual_seed(0)

model = nn.Sequential(
    nn.Conv2d(3, 64, 7, stride=2, padding=3), nn.ReLU(),
    nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, 1000),
).to(device).half().eval()

with torch.no_grad():
    for _ in range(5):
        model(torch.randn(1, 3, 224, 224, device=device, dtype=torch.float16))
    torch.cuda.synchronize()


## 2 — Load generator

The load generator measures end-to-end request latency from the client thread: queueing + model execution + scatter. CUDA synchronization is explicit so CPU timing does not stop at asynchronous kernel submission.


In [ ]:
def percentile(xs, q):
    xs = sorted(xs)
    if not xs:
        raise ValueError("empty sample")
    idx = min(len(xs)-1, max(0, round((len(xs)-1)*q)))
    return xs[idx]

def run_load_test(handler, n_requests=200, concurrency=16):
    work = queue.Queue()
    for i in range(n_requests): work.put(i)
    latencies = []
    lock = threading.Lock()
    def client():
        while True:
            try: work.get_nowait()
            except queue.Empty: return
            x = torch.randn(1, 3, 224, 224, device=device, dtype=torch.float16)
            t0 = time.perf_counter()
            handler(x)
            torch.cuda.synchronize()
            dt_ms = (time.perf_counter() - t0) * 1e3
            with lock: latencies.append(dt_ms)
    threads = [threading.Thread(target=client) for _ in range(concurrency)]
    t0 = time.perf_counter()
    for t in threads: t.start()
    for t in threads: t.join()
    elapsed = time.perf_counter() - t0
    return {"qps": n_requests / elapsed, "p50_ms": percentile(latencies,.50), "p95_ms": percentile(latencies,.95), "p99_ms": percentile(latencies,.99), "samples": len(latencies)}


## 3 — Batcher with instrumentation

The batcher records realized batch size, queue wait, executed batches, and configured bounds so a throughput result can be explained.


In [ ]:
@dataclass
class Request:
    tensor: torch.Tensor
    enqueued_at: float
    done: threading.Event = field(default_factory=threading.Event)
    output: Any = None

class DynamicBatcher:
    def __init__(self, model, max_batch_size=16, max_queue_delay_ms=2.0):
        self.model = model
        self.max_batch_size = int(max_batch_size)
        self.max_delay_s = max_queue_delay_ms / 1000.0
        self.q = queue.Queue()
        self.stop = threading.Event()
        self.batch_sizes = []
        self.queue_wait_ms = []
        self.worker = threading.Thread(target=self._loop, daemon=True)
        self.worker.start()
    def infer(self, x):
        r = Request(x, time.perf_counter())
        self.q.put(r); r.done.wait(); return r.output
    def close(self):
        self.stop.set(); self.worker.join(timeout=1.0)
    def _loop(self):
        while not self.stop.is_set():
            try: first = self.q.get(timeout=0.05)
            except queue.Empty: continue
            batch = [first]
            deadline = first.enqueued_at + self.max_delay_s
            while len(batch) < self.max_batch_size:
                remaining = deadline - time.perf_counter()
                if remaining <= 0: break
                try: batch.append(self.q.get(timeout=remaining))
                except queue.Empty: break
            self.batch_sizes.append(len(batch))
            self.queue_wait_ms.append((time.perf_counter()-first.enqueued_at)*1e3)
            inputs = torch.cat([r.tensor for r in batch], dim=0)
            with torch.no_grad(): outputs = self.model(inputs)
            for i,r in enumerate(batch): r.output=outputs[i:i+1]; r.done.set()


## 4 — The useful failure

Recorded first attempt:

```text
naive                    1438.9 QPS
batch max=32, conc=16    1106.6 QPS   ← regression
```

The first batching configuration was **23% slower than naive serving**. With only 16 concurrent clients, `max_batch=32` cannot fill from one request wave; a 10 ms queue-delay budget can become pure waiting time.


In [ ]:
from pathlib import Path
import json
recorded = json.loads(Path("../data/dynamic_batching_recorded_run.json").read_text())
b = recorded["baseline"]; f = recorded["bad_first_attempt"]
print(f"QPS change: {(f['qps']/b['qps']-1)*100:.1f}%")


## 5 — Sweep the control variable instead of defending the first guess

| max batch | QPS | p50 ms | p99 ms |
|---:|---:|---:|---:|
| 1 | 1,640.6 | 8.6 | 14.3 |
| 4 | 2,503.9 | 3.2 | 21.1 |
| **8** | **2,923.3** | 2.4 | 19.5 |
| 16 | 2,900.9 | **2.2** | **18.5** |
| 32 | 1,134.0 | 11.3 | 20.1 |
| 64 | 1,118.9 | 11.5 | 20.6 |

`max_batch=8` produced the highest measured QPS, **1.78×** the batch=1 run. Larger values collapsed because the workload could not fill them efficiently.


In [ ]:
rows = recorded["sweep"]
best = max(rows, key=lambda r:r["qps"])
base = next(r for r in rows if r["max_batch"]==1)
print(f"best measured max_batch: {best['max_batch']}")
print(f"speedup vs batch=1: {best['qps']/base['qps']:.2f}x")


## 6 — Pareto frontier, not a single magic batch size

Throughput and tail latency are separate objectives. A production decision should remove configurations that are dominated on both axes.


In [ ]:
def pareto_frontier(rows):
    keep=[]
    for a in rows:
        dominated=False
        for b in rows:
            if a is b: continue
            no_worse=b['qps']>=a['qps'] and b['p99_ms']<=a['p99_ms']
            strictly=b['qps']>a['qps'] or b['p99_ms']<a['p99_ms']
            if no_worse and strictly: dominated=True; break
        if not dominated: keep.append(a)
    return sorted(keep,key=lambda r:r['max_batch'])
pareto_frontier(rows)


## 7 — What this establishes

The transferable result is not that batch size 8 is universally best. It is:

```text
batching gain = accelerator utilization recovered
               - queueing delay introduced
               - synchronization / scatter overhead
```

Configured max batch must be evaluated against offered concurrency and arrival rate.
